# 600 · Architecture & design patterns — procedural layer

**Mnemonic: SIX sides of the hexagon — hexagonal architecture.**

Codes covered: **603** Builder · **612** Decorator · **621** Observer · **631** Template method · **645** Dependency rule · **654** Constructor injection · **665** Unidirectional data flow · **674** Event sourcing · **683** Backward compatibility · **692** Value object

**Method:** for every code cell — read it, PREDICT the exact output, run it, compare. A misprediction is the lesson: reread the definition, fix your mental model, run again. Cells run top to bottom in a single kernel; only the marked cell in 683 is allowed to raise.

## 603 · Builder

Constructs a complex object step by step through a fluent or staged API, separating construction from the final representation.

*A Subway sandwich artist: bread? cheese? toasted? Each shout adds one layer, and only the final 'wrap it up!' (build) hands you the finished sub.*

**Watch:** every method returns `self`, so the calls chain — and each shout prints as it lands. Nothing finished exists until `.build()`, which hands back an immutable tuple.

In [1]:
class ConfigBuilder:
    def __init__(self):
        self._parts = {}
        print("ConfigBuilder()            # empty counter, nothing built yet")
    def host(self, h):
        self._parts["host"] = h; print(f"  .host({h!r})"); return self
    def port(self, p):
        self._parts["port"] = p; print(f"  .port({p})"); return self
    def tls(self):
        self._parts["tls"] = True; print("  .tls()"); return self
    def build(self):
        print("  .build()                 # wrap it up!")
        return tuple(sorted(self._parts.items()))   # immutable final object

config = ConfigBuilder().host("db.local").port(5432).tls().build()
print("final immutable config:", config)

ConfigBuilder()            # empty counter, nothing built yet
  .host('db.local')
  .port(5432)
  .tls()
  .build()                 # wrap it up!
final immutable config: (('host', 'db.local'), ('port', 5432), ('tls', True))


## 612 · Decorator

Wraps an object behind the SAME interface to add responsibilities dynamically; wrappers stack in layers.

*A Christmas tree gaining lights, then tinsel, then baubles — at every layer it still answers to 'tree', just fancier each time you wrap it.*

**Watch:** every layer exposes the same `.write()`. The call travels inward to the plain `Writer`, then each wrapper transforms the result on the way back out — so STACKING ORDER matters. Predict both outputs before running the second cell.

In [2]:
class Writer:
    def write(self, text):
        return text

class UpperCaseWriter:                    # same .write() interface, adds SHOUTING
    def __init__(self, inner): self.inner = inner
    def write(self, text): return self.inner.write(text).upper()

class ExclaimWriter:                      # same .write() interface, adds enthusiasm
    def __init__(self, inner): self.inner = inner
    def write(self, text): return self.inner.write(text) + " ... wow!"

stack = ExclaimWriter(UpperCaseWriter(Writer()))
print("Exclaim(Upper(Writer)):", stack.write("hello world"))

Exclaim(Upper(Writer)): HELLO WORLD ... wow!


In [3]:
swapped = UpperCaseWriter(ExclaimWriter(Writer()))
print("Upper(Exclaim(Writer)):", swapped.write("hello world"))
print("same three layers, different order -> different output: order matters")

Upper(Exclaim(Writer)): HELLO WORLD ... WOW!
same three layers, different order -> different output: order matters


## 621 · Observer

A subject keeps a list of subscribers and notifies them automatically when its state changes — one-to-many, publisher pushes.

*A lighthouse blinks once and every ship's bell on the horizon rings itself — each ship had signed the lighthouse's ledger to be notified.*

**Watch:** one `notify()` fans out to every subscribed ship. After Borealis unsubscribes, the second blink reaches only Aurora — predict each ship's final `received` list.

In [4]:
class Ship:
    def __init__(self, name): self.name, self.received = name, []
    def on_event(self, event): self.received.append(event)

class Lighthouse:
    def __init__(self): self._ledger = []
    def subscribe(self, ship): self._ledger.append(ship)
    def unsubscribe(self, ship): self._ledger.remove(ship)
    def notify(self, event):
        print(f"lighthouse emits {event!r} -> {len(self._ledger)} ship(s)")
        for ship in self._ledger: ship.on_event(event)

aurora, borealis = Ship("Aurora"), Ship("Borealis")
light = Lighthouse()
light.subscribe(aurora); light.subscribe(borealis)
light.notify("blink-1")
light.unsubscribe(borealis)
light.notify("blink-2")
print("Aurora   received:", aurora.received)
print("Borealis received:", borealis.received)

lighthouse emits 'blink-1' -> 2 ship(s)
lighthouse emits 'blink-2' -> 1 ship(s)
Aurora   received: ['blink-1', 'blink-2']
Borealis received: ['blink-1']


## 631 · Template method

A base-class method fixes an algorithm's skeleton and defers specific steps to subclass hook overrides.

*A Mad Libs card: the plot is printed in permanent ink; subclasses may only scribble words into the blanks, then the card reads itself aloud.*

**Watch:** `run()` is never overridden — load → process → report happens in permanent ink for both subclasses. Only the `process()` blank differs. Predict both report lines.

In [5]:
class Pipeline:
    def run(self):                        # the skeleton, printed in permanent ink
        data = self.load()
        result = self.process(data)      # the blank subclasses fill in
        self.report(data, result)
    def load(self): return [3, 1, 2]
    def process(self, data): raise NotImplementedError("subclass fills the blank")
    def report(self, data, result):
        print(f"{type(self).__name__}: {data} -> {result}")

class SortPipeline(Pipeline):
    def process(self, data): return sorted(data)

class SumPipeline(Pipeline):
    def process(self, data): return sum(data)

SortPipeline().run()
SumPipeline().run()

SortPipeline: [3, 1, 2] -> [1, 2, 3]
SumPipeline: [3, 1, 2] -> 6


## 645 · Dependency rule

Source-code dependencies may only point inward: inner circles know nothing — not even names — of anything in outer circles.

*A one-way pilgrimage road spiraling into a mountain shrine: carts roll inward only, and the monks inside have never once heard the market's name.*

**Watch:** `compute_invoice` is pure — no imports, no I/O, no idea who calls it. First it runs alone (testable in isolation); then two different adapters call INWARD to it and get the identical answer.

In [6]:
# THE DOMAIN CORE — pure function, zero I/O, knows nothing of the outside world
def compute_invoice(items):
    subtotal = sum(qty * price for _name, qty, price in items)
    tax = round(subtotal * 0.20, 2)
    return {"subtotal": subtotal, "tax": tax, "total": round(subtotal + tax, 2)}

# testable completely alone — no CLI, no DB, no framework required:
print("domain alone   ->", compute_invoice([("book", 2, 10.0), ("pen", 5, 1.5)]))

domain alone   -> {'subtotal': 27.5, 'tax': 5.5, 'total': 33.0}


In [7]:
# THE OUTER RING — adapters depend on (call) the domain; the domain never calls back
def cli_adapter(argv):                                 # port 1: command line
    items = [(n, int(q), float(p)) for n, q, p in (a.split(":") for a in argv)]
    print("CLI adapter    ->", compute_invoice(items))

def fake_db_adapter():                                 # port 2: persistence
    rows = [("book", 2, 10.0), ("pen", 5, 1.5)]        # pretend SELECT result
    print("fake-DB adapter->", compute_invoice(rows))

cli_adapter(["book:2:10.0", "pen:5:1.5"])
fake_db_adapter()
print("identical result under both adapters — the shrine never heard the market's name")

CLI adapter    -> {'subtotal': 27.5, 'tax': 5.5, 'total': 33.0}
fake-DB adapter-> {'subtotal': 27.5, 'tax': 5.5, 'total': 33.0}
identical result under both adapters — the shrine never heard the market's name


## 654 · Constructor injection

Dependencies arrive as constructor parameters, making them mandatory, visible, and enabling immutable, always-valid objects.

*A spaceship that physically cannot launch until every crew member is strapped in at ignition — and the hatch welds itself shut after liftoff (immutability).*

**Watch:** `OrderService` cannot exist without a repo — it's strapped in at `__init__`. The second cell swaps in a `FakeRepo` that only records calls: the service code never changes, and the fake's call log makes it trivially testable.

In [8]:
class OrderService:
    def __init__(self, repo):             # dependency strapped in at ignition
        self._repo = repo                 # no launch without it
    def place(self, order_id, item):
        self._repo.save(order_id, item)
        return f"order {order_id} placed"

class RealRepo:                           # 'production' repo, dict-backed
    def __init__(self): self._db = {}
    def save(self, key, value): self._db[key] = value

real = RealRepo()
service = OrderService(real)
print(service.place(1, "espresso"))
print("real repo contents:", real._db)

order 1 placed
real repo contents: {1: 'espresso'}


In [9]:
class FakeRepo:                           # test double — records calls, stores nothing
    def __init__(self): self.calls = []
    def save(self, key, value): self.calls.append(("save", key, value))

fake = FakeRepo()
service = OrderService(fake)              # same service class, swapped dependency
print(service.place(2, "latte"))
print(service.place(3, "mocha"))
print("fake's call log:", fake.calls)
print("OrderService unchanged — the constructor made the swap trivial")

order 2 placed
order 3 placed
fake's call log: [('save', 2, 'latte'), ('save', 3, 'mocha')]
OrderService unchanged — the constructor made the swap trivial


## 665 · Unidirectional data flow

State flows down as props and changes flow up as events or actions through one loop, making every mutation traceable.

*A one-way water park: state pours down the slide into the views below; riders send wishes back up only via the message-balloon chimney (actions). No swimming upstream.*

**Watch:** the reducer never mutates — it returns a NEW state for each action, and every transition prints, so the whole history is traceable. Predict the state after each of the four actions.

In [10]:
def reducer(state, action):               # (old state, action) -> NEW state
    kind, payload = action
    if kind == "add":    return state + [payload]
    if kind == "remove": return [x for x in state if x != payload]
    if kind == "rename":
        old, new = payload
        return [new if x == old else x for x in state]
    return state

actions = [("add", "draft"), ("add", "todo"),
           ("rename", ("draft", "final")), ("remove", "todo")]
state = []
for action in actions:
    state = reducer(state, action)        # one loop, one direction
    print(f"{str(action):32} -> {state}")
print("final state:", state)

('add', 'draft')                 -> ['draft']
('add', 'todo')                  -> ['draft', 'todo']
('rename', ('draft', 'final'))   -> ['final', 'todo']
('remove', 'todo')               -> ['final']
final state: ['final']


## 674 · Event sourcing

Persist every state change as an immutable event; current state is derived by replaying the log, which is the system of record.

*An accountant's ink ledger: nobody erases, ever; a mistake gets a correcting line, and today's balance is re-summed from page one whenever anyone asks.*

**Watch:** state is a fold over the log — replaying the same log yields the same balance, every time. The second cell appends one new event (the only legal write) and re-folds: history and state are printed side by side.

In [11]:
events = [("deposited", 100), ("withdrawn", 30), ("deposited", 50)]   # append-only log

def fold(log):
    balance = 0
    for kind, amount in log:
        balance += amount if kind == "deposited" else -amount
    return balance

print("state from fold:   ", fold(events))
print("REPLAY (same log): ", fold(events), "  <- replaying always rebuilds the same state")

state from fold:    120
REPLAY (same log):  120   <- replaying always rebuilds the same state


In [12]:
events.append(("withdrawn", 20))          # append-only: never edit, never erase
print("history (the system of record):")
for i, event in enumerate(events):
    print(f"  page {i}: {event}")
print("current state (re-folded):", fold(events))
print("the log IS the data; state is just a projection of it")

history (the system of record):
  page 0: ('deposited', 100)
  page 1: ('withdrawn', 30)
  page 2: ('deposited', 50)
  page 3: ('withdrawn', 20)
current state (re-folded): 100
the log IS the data; state is just a projection of it


## 683 · Backward compatibility

Evolve APIs additively — add optional fields, never rename, remove, or retype — while clients ignore unknown fields (tolerant reader).

*Grandma's recipe card with extra sticky notes stapled on: old cooks simply ignore the staples and the cake still rises — but no one may ever cross out a line.*

**Watch:** v2 staples on `email` and the old client keeps working, blissfully ignoring it. v3 crosses out a line (renames `name` → `full_name`) — the next cell INTENTIONALLY raises: read the traceback and see exactly which client broke and why.

In [13]:
def api_v1(user_id):
    return {"name": "Ada"}

def api_v2(user_id):                      # additive change: a sticky note stapled on
    return {"name": "Ada", "email": "ada@example.com"}

def old_client(api):                      # written against v1, reads only 'name'
    return f"Hello, {api(42)['name']}!"

print("old client on v1:", old_client(api_v1))
print("old client on v2:", old_client(api_v2), " <- unknown field ignored, still works")

old client on v1: Hello, Ada!
old client on v2: Hello, Ada!  <- unknown field ignored, still works


In [14]:
# INTENDED ERROR — read the traceback
def api_v3(user_id):                      # BREAKING change: renamed name -> full_name
    return {"full_name": "Ada Lovelace"}

print("old client on v3:")
print(old_client(api_v3))                 # KeyError: 'name' — renames break old clients

old client on v3:


KeyError: 'name'

## 692 · Value object

An immutable object defined entirely by its attribute values, with no identity; equal fields mean interchangeable objects.

*Two identical 2x4 red LEGO bricks: shuffle them behind your back and nobody can ever tell. To 'change' one's color you don't repaint — you pick up a different brick.*

**Watch:** two `Money` objects with equal fields are `==` and mutation is physically refused (`FrozenInstanceError`, caught in-cell). The second cell shows the contrast: `User` ENTITIES compare by `id` — same name but different ids are `!=`, and renaming an entity doesn't change who it is.

In [15]:
from dataclasses import dataclass, FrozenInstanceError

@dataclass(frozen=True)
class Money:                              # value object: all fields, no identity
    amount: int
    currency: str

a, b = Money(10, "EUR"), Money(10, "EUR")
print("Money(10,'EUR') == Money(10,'EUR') ->", a == b, " (equal fields = interchangeable)")
try:
    a.amount = 99                         # you don't repaint a brick...
except FrozenInstanceError as e:
    print("mutation blocked:", type(e).__name__, "-", e)
print("...you pick up a different brick:", Money(99, "EUR"))

Money(10,'EUR') == Money(10,'EUR') -> True  (equal fields = interchangeable)
mutation blocked: FrozenInstanceError - cannot assign to field 'amount'
...you pick up a different brick: Money(amount=99, currency='EUR')


In [16]:
class User:                               # entity: defined by identity, not fields
    def __init__(self, uid, name): self.id, self.name = uid, name
    def __eq__(self, other): return isinstance(other, User) and self.id == other.id
    def __repr__(self): return f"User(id={self.id}, name={self.name!r})"

u1, u2 = User(1, "Ada"), User(2, "Ada")
print(u1, "==", u2, "->", u1 == u2, " (same name, different identity)")
u1.name = "Ada Lovelace"                  # entities may mutate over their lifetime
print("after rename:", u1, "==", User(1, "Ada"), "->", u1 == User(1, "Ada"))
print("same id -> still the same entity, whatever the fields now say")

User(id=1, name='Ada') == User(id=2, name='Ada') -> False  (same name, different identity)
after rename: User(id=1, name='Ada Lovelace') == User(id=1, name='Ada') -> True
same id -> still the same entity, whatever the fields now say
